In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
dbutils.widgets.text("catalog_name", "olist_catalog", "Catalog")
dbutils.widgets.text("schema_name", "bronze", "Schema")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
schema_name  = dbutils.widgets.get("schema_name")

In [0]:
%sql
USE CATALOG IDENTIFIER(:catalog_name);
USE SCHEMA IDENTIFIER(:schema_name);

In [0]:
catalog_df = spark.sql("SELECT current_catalog() AS catalog")
display(catalog_df)

In [0]:
%sql
    
CREATE TABLE orders AS
SELECT * FROM
  read_files(
    '/Volumes/olist_catalog/bronze/olist_all/olist_orders_dataset.csv',
    format => 'csv',
    header => true
  )

In [0]:
%sql
CREATE TABLE customers AS
SELECT * FROM
  read_files(
    '/Volumes/olist_catalog/bronze/olist_all/olist_customers_dataset.csv',
    format => 'csv',
    header => true
  )

In [0]:
## TODO: run a loop to check the volume location for files ending in '_dataset.csv' and take the word before and save it as table with the word as the name. if it doesn't end like _dataset then don't
# probably using dbutils.ls saved to a list or something

In [0]:
files_in_volume = dbutils.fs.ls('/Volumes/olist_catalog/bronze/olist_all/')
files_in_volume = [f.path for f in files_in_volume if f.name.endswith('_dataset.csv')]
print(files_in_volume)

In [0]:
for path in files_in_volume:
    file_name = path.split("/")[-1]  # e.g. olist_customers_dataset.csv

    # Strip suffix and prefix: olist_customers_dataset.csv -> customers
    base = file_name.replace("_dataset.csv", "")   # olist_customers
    table_name = base.replace("olist_", "")        # customers

    sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} AS
    SELECT * FROM read_files(
      '{path}',
      format => 'csv',
      header => true
    )
    """

    print(f"Creating table {table_name} from {path}")
    spark.sql(sql)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS product_category_name_translation AS
SELECT * FROM
  read_files(
    '/Volumes/olist_catalog/bronze/olist_all/product_category_name_translation.csv',
    format => 'csv',
    header => true
  )

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SELECT * FROM products

In [0]:
%sql
SELECT * FROM product_category_name_translation

In [0]:
%sql
CREATE OR REPLACE TABLE products_translated AS
SELECT 
  prod.* EXCEPT (product_category_name),
  trans.product_category_name_english AS product_category_name
FROM products AS prod
INNER JOIN product_category_name_translation AS trans
ON prod.product_category_name = trans.product_category_name;
    
SELECT * FROM products_translated;